# Google Colab Pro GPU SSH Setup & Verification

This notebook configures SSH access and verifies GPU hardware on your active Google Colab runtime.

### Runtime Hardware Check
Ensure you have selected a GPU accelerator: **Runtime > Change runtime type > T4 / L4 / A100**.

In [ ]:
# Verify GPU availability and system info
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## Method 1: Tailscale SSH (Direct Private Tailnet Mesh)

Since your local machine is already running Tailscale, this connects your Colab runtime directly into your private Tailnet.

1. Go to your [Tailscale Admin Console: Settings > Keys](https://login.tailscale.com/admin/settings/keys)
2. Generate an **Auth Key** (check *Reusable* or *Ephemeral*).
3. Paste the key below and run the cell.

In [ ]:
TAILSCALE_AUTH_KEY = "tskey-auth-PASTE_YOUR_KEY_HERE"  # <-- Replace with your auth key

# Install Tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

# Start daemon in userspace networking mode (required for unprivileged Colab containers)
!nohup tailscaled --tun=userspace-networking > /dev/null 2>&1 &

import time
time.sleep(3)

# Bring up Tailscale with SSH enabled
!tailscale up --authkey={TAILSCALE_AUTH_KEY} --ssh --hostname=colab-gpu

# Print IP for connection
!tailscale ip -4

### Connecting via Tailscale
From your local terminal:
```bash
ssh root@colab-gpu
# or:
tailscale ssh root@colab-gpu
```

## Method 2: Cloudflare Tunnel / tmate (Public Key SSH)

If you prefer a direct public key connection using Cloudflare Tunnel:

In [ ]:
# Paste your public SSH key (~/.ssh/id_ed25519.pub)
PUBLIC_KEY = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAICj1OMZcSfJKgKlvTk5ZWSMnWvOhy6q8A5QDS7bGN8Ct gsfochesatto@alaska.edu"

!mkdir -p /root/.ssh
with open('/root/.ssh/authorized_keys', 'w') as f:
    f.write(PUBLIC_KEY + '\n')
!chmod 700 /root/.ssh && chmod 600 /root/.ssh/authorized_keys

# Install & start OpenSSH server
!apt-get update -qq && apt-get install -y -qq openssh-server
!service ssh start

# Download cloudflared and start tunnel
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!nohup cloudflared tunnel --url ssh://localhost:22 > /tmp/tunnel.log 2>&1 &

import time, re
time.sleep(5)
with open('/tmp/tunnel.log') as f:
    log = f.read()
urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log)
if urls:
    domain = urls[0].replace('https://', '')
    print(f"\nTunnel Domain: {domain}")
    print("\nAdd this to your local ~/.ssh/config:")
    print(f"Host colab-cf\n    HostName {domain}\n    User root\n    ProxyCommand cloudflared access ssh --hostname %h\n")
    print("Then connect locally with: ssh colab-cf")
else:
    print("Could not retrieve tunnel URL. Log content:")
    print(log)